# BAMO — G4 session 1: setup check, speed and concurrency probe

Checks the code and data on the Colab G4, runs the test suite on the GPU, measures training speed (T_ref estimate) and how many training processes one G4 can run at once. About 20–30 minutes (≈ 3–5 compute units). At the end, copy the printed **SUMMARY** back to Claude.

**In VS Code:** choose **Select Kernel → Colab → New Colab Server → G4**, then **Run All**. If the first cell reports that Google Drive is not mounted, run the Command Palette command **Colab: Mount Google Drive to Server...**, run the cell it adds, then **Run All** again.

Results are written to **My Drive/bamo_runs**. Disconnect the runtime when finished: G4 bills while idle.

In [ ]:
import os
OUTPUT = '/content/drive/MyDrive/bamo_runs'
if not os.path.ismount('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as error:
        raise RuntimeError(
            'Google Drive is not mounted. In VS Code, open the Command Palette and run '
            '"Colab: Mount Google Drive to Server...", run the cell it adds, then rerun this cell. '
            f'({error})')
assert os.path.ismount('/content/drive') and os.path.isdir('/content/drive/MyDrive'), 'Drive mount failed'
os.makedirs(OUTPUT, exist_ok=True)
print('Results will be saved in Google Drive:', OUTPUT.replace('/content/drive/', ''))

Embedded, hash-checked code bundle (67 files: `src/`, `scripts/`, `configs/`, `tests/`). Do not edit; rebuild with `scripts/build_colab_notebook.py`.

In [ ]:
BUNDLE_B64 = '...'

In [ ]:
import base64, hashlib, io, json, pathlib, subprocess, sys, zipfile
bundle = base64.b64decode(BUNDLE_B64)
BUNDLE_SHA = 'c4c8efd3c0456f69c497d249cb274ac14cb335d0cc86c792a9304b4607ae0ae6'
assert hashlib.sha256(bundle).hexdigest() == BUNDLE_SHA, 'embedded code bundle is corrupt'
root = pathlib.Path('/content/bamo')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle)) as source:
    for name in source.namelist():
        assert not name.startswith('/') and '..' not in pathlib.PurePosixPath(name).parts
    source.extractall(root)
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *'wfdb==4.3.1 optuna==5.0.0 dehb==0.1.1 ConfigSpace==1.2.2 pytest'.split()], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
sys.path.insert(0, str(root / 'src'))
import torch
assert torch.cuda.is_available(), 'Select a Colab GPU kernel (G4) before running'
environment = pathlib.Path(OUTPUT) / 'environment'
environment.mkdir(exist_ok=True)
(environment / 'nvidia_smi.txt').write_text(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
(environment / 'pip_freeze.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
(environment / 'bundle_sha256.txt').write_text(BUNDLE_SHA)
memory_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print(f'Code bundle verified ({BUNDLE_SHA[:12]}...). GPU: {torch.cuda.get_device_name()}, '
      f'{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB; '
      f'{os.cpu_count()} CPUs; {memory_gb:.0f} GiB RAM; torch {torch.__version__}')

## Data
Download PTB-XL v1.0.3 (metadata + 100 Hz records) from PhysioNet's S3 mirror, verify every file against PhysioNet's SHA256SUMS, rebuild the prepared arrays, and check they are byte-identical to the local preparation.

In [ ]:
import time
EXPECTED = {"signals_100hz.npy": "630948cdb659b3f8cdab76d993f895de2073bfb7b59f512dbc7cceefed649a09", "labels.npy": "3a95ec7122f28b6e2e70b3d799c8e16c6a12f7bd53f8f11d557bd4a079810cf5"}
started = time.time()
for step in (['scripts/download_ptbxl.py', '--workers', '64'], ['scripts/prepare_data.py']):
    result = subprocess.run([sys.executable, *step], capture_output=True, text=True)
    print(result.stdout[-1500:])
    assert result.returncode == 0, result.stderr[-3000:]
from bamo.utils import sha256_file
for name, digest in EXPECTED.items():
    assert sha256_file(f'data/processed/{name}') == digest, f'{name} differs from the local preparation'
print(f'Data ready and identical to the local arrays in {time.time() - started:.0f} s')

## Tests

In [ ]:
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:warnings'], capture_output=True, text=True)
print(result.stdout[-2000:])
assert result.returncode == 0, 'tests failed'

## Speed check (single process)

In [ ]:
result = subprocess.run([sys.executable, 'scripts/run_pilot.py', 'speed', '--out-root', OUTPUT, '--cu-per-hour', '8.9'],
                        capture_output=True, text=True)
print(result.stdout[-4000:])
assert result.returncode == 0, result.stderr[-3000:]

## Concurrency probe (K training processes sharing the G4)

In [ ]:
result = subprocess.run([sys.executable, 'scripts/concurrency_probe.py', '--levels', '1', '4', '8', '16', '24',
                         '--out', f'{OUTPUT}/pilot/concurrency.json'], capture_output=True, text=True)
print(result.stdout[-4000:])
assert result.returncode == 0, result.stderr[-3000:]

## Summary (copy this back)

In [ ]:
speed = json.loads(pathlib.Path(f'{OUTPUT}/pilot/p1_speed.json').read_text())
conc = json.loads(pathlib.Path(f'{OUTPUT}/pilot/concurrency.json').read_text())
s = speed['summary']
print('SUMMARY')
print('GPU:', speed['hardware'].get('gpu'), '| torch', speed['hardware'].get('torch'), '| bundle', BUNDLE_SHA[:12])
print(f"single process: median {s['median_s_per_epoch']:.2f} s/epoch (min {s['min']:.2f}, max {s['max']:.2f}); "
      f"T_ref ~ {s['t_ref_estimate_s']:.0f} s")
for design, p in s['projection'].items():
    print(f"  {design}: {p['gpu_hours']:.0f} GPU-h sequential, ~{p['compute_units']:.0f} CU")
for probe, rows in conc['probes'].items():
    print(probe, ' '.join(f"K{r['k']}:x{r['throughput_gain']:.1f}" for r in rows))